In [1]:
#!/usr/bin/env python3
# =========================================================
# ENSEMBLE SUMMARIZATION SYSTEM
# Three Fine-tuned Models: BART + LED + PEGASUS
# With InLegalBERT MMR Sentence Selection
# =========================================================

import os
import json
import torch
import numpy as np
from tqdm import tqdm
from nltk.tokenize import sent_tokenize
from sklearn.metrics.pairwise import cosine_similarity
from transformers import (
    AutoTokenizer, 
    AutoModel, 
    AutoModelForSeq2SeqLM,
    LEDForConditionalGeneration,
    PegasusForConditionalGeneration
)
from collections import Counter
import evaluate
import nltk

# Download NLTK data
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

# Device setup
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🚀 Device: {device}")


🚀 Device: cuda


In [2]:

# =========================================================
# CONFIGURATION
# =========================================================

# Model Paths
MODEL_PATHS = {
    "BART": "BART",
    "LED": "LED",
    "PEGASUS": "PEGASUS"
}

# Data paths
TEST_JSON_PATH = "test.json"
OUTPUT_DIR = "./ensemble_results"

# Model-specific parameters
MODEL_CONFIG = {
    "BART": {
        "max_input": 1024,
        "max_output": 256,
        "num_beams": 4
    },
    "LED": {
        "max_input": 4096,  # LED handles longer inputs
        "max_output": 512,
        "num_beams": 4
    },
    "PEGASUS": {
        "max_input": 1024,
        "max_output": 256,
        "num_beams": 4
    }
}

# MMR parameters
MMR_K = 60  # For BART and PEGASUS
MMR_K_LED = 200  # LED can handle more sentences
MMR_LAMBDA = 0.7

# Ensemble weights (adjust based on validation performance)
ENSEMBLE_WEIGHTS = {
    "BART": 0.35,
    "LED": 0.30,
    "PEGASUS": 0.35
}

# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"📂 Output directory: {OUTPUT_DIR}")
print(f"📂 Test data: {TEST_JSON_PATH}")


📂 Output directory: ./ensemble_results
📂 Test data: test.json


In [3]:

# =========================================================
# LOAD InLegalBERT FOR SENTENCE EMBEDDINGS
# =========================================================

print("\n📚 Loading InLegalBERT...")

legal_model_name = "law-ai/InLegalBERT"
legal_tokenizer = AutoTokenizer.from_pretrained(legal_model_name)
legal_model = AutoModel.from_pretrained(legal_model_name).to(device)
legal_model.eval()

print("✅ InLegalBERT loaded successfully")

@torch.no_grad()
def embed_with_legalbert(texts, batch_size=16):
    """Compute sentence embeddings using InLegalBERT."""
    if len(texts) == 0:
        return np.array([])
    
    embs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        
        enc = legal_tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt"
        ).to(device)
        
        out = legal_model(**enc).last_hidden_state
        mask = enc["attention_mask"].unsqueeze(-1).float()
        pooled = (out * mask).sum(1) / mask.sum(1)
        
        embs.append(pooled.cpu().numpy())
    
    return np.vstack(embs)

# =========================================================
# MMR SENTENCE SELECTION
# =========================================================

def select_sentences_mmr(judgment, k=60, lambda_param=0.7):
    """Select top-k sentences using MMR algorithm."""
    sentences = sent_tokenize(judgment)
    n_original = len(sentences)
    
    if len(sentences) <= k:
        return judgment, sentences, {
            "method": "no_filtering",
            "original_sents": n_original,
            "selected_sents": n_original
        }
    
    embeddings = embed_with_legalbert(sentences)
    centroid = embeddings.mean(axis=0, keepdims=True)
    relevance = cosine_similarity(embeddings, centroid).squeeze()
    
    selected_indices = [int(np.argmax(relevance))]
    remaining_indices = set(range(len(sentences))) - set(selected_indices)
    
    while len(selected_indices) < k and remaining_indices:
        mmr_scores = []
        
        for i in remaining_indices:
            sim_to_selected = cosine_similarity(
                embeddings[i:i+1], 
                embeddings[selected_indices]
            ).max()
            
            mmr_score = (lambda_param * relevance[i] - 
                        (1 - lambda_param) * sim_to_selected)
            
            mmr_scores.append((i, mmr_score))
        
        best_idx = max(mmr_scores, key=lambda x: x[1])[0]
        selected_indices.append(best_idx)
        remaining_indices.remove(best_idx)
    
    selected_indices.sort()
    selected_sentences = [sentences[i] for i in selected_indices]
    filtered_text = " ".join(selected_sentences)
    
    selection_info = {
        "method": "mmr",
        "original_sents": n_original,
        "selected_sents": len(selected_indices),
        "compression_ratio": len(selected_indices) / n_original,
        "lambda": lambda_param
    }
    
    return filtered_text, selected_sentences, selection_info

print("✅ MMR sentence selection ready")

# =========================================================
# LOAD ALL THREE MODELS
# =========================================================

print("\n📚 Loading all three fine-tuned models...")

models = {}
tokenizers = {}



📚 Loading InLegalBERT...


/home/RSlab/.conda/envs/legal-nlp/lib/python3.10/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


✅ InLegalBERT loaded successfully
✅ MMR sentence selection ready

📚 Loading all three fine-tuned models...


In [4]:

# Load BART
print("\n  [1/3] Loading BART...")
tokenizers["BART"] = AutoTokenizer.from_pretrained(MODEL_PATHS["BART"])
models["BART"] = AutoModelForSeq2SeqLM.from_pretrained(MODEL_PATHS["BART"]).to(device)
models["BART"].eval()
print(f"  ✅ BART loaded - {models['BART'].num_parameters():,} parameters")

# Load LED
print("\n  [2/3] Loading LED...")
tokenizers["LED"] = AutoTokenizer.from_pretrained(MODEL_PATHS["LED"])
models["LED"] = LEDForConditionalGeneration.from_pretrained(MODEL_PATHS["LED"]).to(device)
models["LED"].eval()
print(f"  ✅ LED loaded - {models['LED'].num_parameters():,} parameters")

# Load PEGASUS
print("\n  [3/3] Loading PEGASUS...")
tokenizers["PEGASUS"] = AutoTokenizer.from_pretrained(MODEL_PATHS["PEGASUS"])
models["PEGASUS"] = PegasusForConditionalGeneration.from_pretrained(MODEL_PATHS["PEGASUS"]).to(device)
models["PEGASUS"].eval()
print(f"  ✅ PEGASUS loaded - {models['PEGASUS'].num_parameters():,} parameters")

print("\n✅ All models loaded successfully!")

# =========================================================
# LOAD TEST DATA
# =========================================================

print(f"\n📂 Loading test data from {TEST_JSON_PATH}...")

with open(TEST_JSON_PATH, 'r', encoding='utf8') as f:
    test_data = json.load(f)

print(f"✅ Loaded {len(test_data)} test samples")

# =========================================================
# EVALUATION METRICS
# =========================================================

print("\n📊 Loading evaluation metrics...")

rouge_metric = evaluate.load("rouge")
bertscore_metric = evaluate.load("bertscore")

print("✅ Metrics loaded: ROUGE, BERTScore")

def compute_metrics(predictions, references):
    """Compute ROUGE and BERTScore metrics."""
    rouge_scores = rouge_metric.compute(
        predictions=predictions,
        references=references,
        use_stemmer=True
    )
    
    bert_scores = bertscore_metric.compute(
        predictions=predictions,
        references=references,
        model_type="roberta-large",
        lang="en",
        device=device,
        batch_size=8
    )
    
    return {
        "rouge1": float(rouge_scores["rouge1"]),
        "rouge2": float(rouge_scores["rouge2"]),
        "rougeL": float(rouge_scores["rougeL"]),
        "bertscore_precision": float(np.mean(bert_scores["precision"])),
        "bertscore_recall": float(np.mean(bert_scores["recall"])),
        "bertscore_f1": float(np.mean(bert_scores["f1"])),
    }

# =========================================================
# SINGLE MODEL SUMMARY GENERATION
# =========================================================

def generate_summary(model_name, judgment, reference):
    """Generate summary using a single model."""
    model = models[model_name]
    tokenizer = tokenizers[model_name]
    config = MODEL_CONFIG[model_name]
    
    # Use appropriate MMR k based on model
    k_value = MMR_K_LED if model_name == "LED" else MMR_K
    
    # Select sentences using MMR
    filtered_text, _, selection_info = select_sentences_mmr(
        judgment, 
        k=k_value, 
        lambda_param=MMR_LAMBDA
    )
    
    # Tokenize
    inputs = tokenizer(
        filtered_text,
        return_tensors="pt",
        truncation=True,
        max_length=config["max_input"]
    ).to(device)
    
    # Generate
    with torch.no_grad():
        summary_ids = model.generate(
            inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_length=config["max_output"],
            num_beams=config["num_beams"],
            early_stopping=True,
            no_repeat_ngram_size=3
        )
    
    summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    
    return summary, selection_info

# =========================================================
# ENSEMBLE FUSION STRATEGIES
# =========================================================

def ensemble_voting(summaries_dict):
    """
    Voting-based ensemble: Select sentences appearing in ≥2 models.
    """
    all_sentences = []
    
    for model_name, summary in summaries_dict.items():
        sents = sent_tokenize(summary)
        all_sentences.extend(sents)
    
    # Count sentence occurrences (normalized)
    sentence_counts = Counter()
    for sent in all_sentences:
        # Normalize: lowercase and strip
        normalized = sent.lower().strip()
        sentence_counts[normalized] += 1
    
    # Select sentences appearing in at least 2 models
    threshold = 2
    selected = [sent for sent, count in sentence_counts.items() if count >= threshold]
    
    # If too few, fall back to most common sentences
    if len(selected) < 3:
        selected = [sent for sent, _ in sentence_counts.most_common(10)]
    
    return " ".join(selected)

def ensemble_weighted_concat(summaries_dict, weights):
    """
    Weighted concatenation: Combine summaries with quality-based weights.
    Simply concatenate and take first N sentences based on weights.
    """
    weighted_parts = []
    
    for model_name, summary in summaries_dict.items():
        weight = weights[model_name]
        sents = sent_tokenize(summary)
        n_sents = max(1, int(len(sents) * weight))
        weighted_parts.extend(sents[:n_sents])
    
    # Remove duplicates while preserving order
    seen = set()
    unique_sents = []
    for sent in weighted_parts:
        normalized = sent.lower().strip()
        if normalized not in seen:
            seen.add(normalized)
            unique_sents.append(sent)
    
    return " ".join(unique_sents)

def ensemble_best_model(summaries_dict, reference):
    """
    Select best single summary based on ROUGE-2 with reference.
    """
    best_score = -1
    best_summary = ""
    
    for model_name, summary in summaries_dict.items():
        score = rouge_metric.compute(
            predictions=[summary],
            references=[reference],
            use_stemmer=True
        )["rouge2"]
        
        if score > best_score:
            best_score = score
            best_summary = summary
    
    return best_summary

def ensemble_sentence_ranking(summaries_dict):
    """
    Rank-based fusion: Score each sentence by appearance position across models.
    """
    sentence_positions = {}
    
    for model_name, summary in summaries_dict.items():
        sents = sent_tokenize(summary)
        for pos, sent in enumerate(sents):
            normalized = sent.lower().strip()
            if normalized not in sentence_positions:
                sentence_positions[normalized] = []
            # Lower position = higher importance
            sentence_positions[normalized].append(pos)
    
    # Score: average position (lower is better)
    sentence_scores = {
        sent: np.mean(positions) 
        for sent, positions in sentence_positions.items()
    }
    
    # Sort by score and take top sentences
    ranked = sorted(sentence_scores.items(), key=lambda x: x[1])
    top_sentences = [sent for sent, _ in ranked[:15]]
    
    return " ".join(top_sentences)

# =========================================================
# GENERATE SUMMARIES FOR ALL MODELS
# =========================================================

print("\n" + "="*70)
print("🚀 GENERATING SUMMARIES WITH ALL THREE MODELS")
print("="*70)

all_model_summaries = {
    "BART": [],
    "LED": [],
    "PEGASUS": []
}

ensemble_summaries = {
    "voting": [],
    "weighted": [],
    "best_model": [],
    "ranking": []
}

references = []

for idx, item in enumerate(tqdm(test_data, desc="Processing samples")):
    judgment = item["judgment"]
    reference = item["reference_summary"]
    references.append(reference)
    
    # Generate summary with each model
    summaries_dict = {}
    
    for model_name in ["BART", "LED", "PEGASUS"]:
        summary, _ = generate_summary(model_name, judgment, reference)
        all_model_summaries[model_name].append(summary)
        summaries_dict[model_name] = summary
    
    # Apply ensemble strategies
    ensemble_summaries["voting"].append(ensemble_voting(summaries_dict))
    ensemble_summaries["weighted"].append(ensemble_weighted_concat(summaries_dict, ENSEMBLE_WEIGHTS))
    ensemble_summaries["best_model"].append(ensemble_best_model(summaries_dict, reference))
    ensemble_summaries["ranking"].append(ensemble_sentence_ranking(summaries_dict))

print("\n✅ All summaries generated!")

# =========================================================
# EVALUATE ALL APPROACHES
# =========================================================

print("\n" + "="*70)
print("📊 EVALUATING ALL APPROACHES")
print("="*70)

results = {}

# Evaluate individual models
for model_name in ["BART", "LED", "PEGASUS"]:
    print(f"\n  Evaluating {model_name}...")
    metrics = compute_metrics(all_model_summaries[model_name], references)
    results[model_name] = metrics
    
    print(f"  ✅ {model_name} - ROUGE-2: {metrics['rouge2']:.4f}, BERTScore F1: {metrics['bertscore_f1']:.4f}")

# Evaluate ensemble strategies
for strategy in ["voting", "weighted", "best_model", "ranking"]:
    print(f"\n  Evaluating Ensemble-{strategy}...")
    metrics = compute_metrics(ensemble_summaries[strategy], references)
    results[f"ensemble_{strategy}"] = metrics
    
    print(f"  ✅ Ensemble-{strategy} - ROUGE-2: {metrics['rouge2']:.4f}, BERTScore F1: {metrics['bertscore_f1']:.4f}")

# =========================================================
# COMPARISON TABLE
# =========================================================

print("\n" + "="*70)
print("📊 COMPREHENSIVE RESULTS COMPARISON")
print("="*70)

print(f"\n{'Approach':<20} {'ROUGE-1':<10} {'ROUGE-2':<10} {'ROUGE-L':<10} {'BERTScore F1':<12}")
print("-" * 70)

for approach_name, metrics in results.items():
    print(f"{approach_name:<20} {metrics['rouge1']:<10.4f} {metrics['rouge2']:<10.4f} "
          f"{metrics['rougeL']:<10.4f} {metrics['bertscore_f1']:<12.4f}")

# Find best approach
best_approach = max(results.items(), key=lambda x: x[1]['rouge2'])
print("\n" + "="*70)
print(f"🏆 BEST APPROACH: {best_approach[0].upper()}")
print(f"   ROUGE-2: {best_approach[1]['rouge2']:.4f}")
print(f"   BERTScore F1: {best_approach[1]['bertscore_f1']:.4f}")
print("="*70)

# =========================================================
# SAVE ALL RESULTS
# =========================================================

print("\n💾 Saving results...")

# Save individual model summaries
for model_name in ["BART", "LED", "PEGASUS"]:
    output_path = os.path.join(OUTPUT_DIR, f"{model_name.lower()}_summaries.json")
    with open(output_path, 'w', encoding='utf8') as f:
        json.dump([
            {
                "id": item.get("id", idx),
                "generated_summary": summary,
                "reference_summary": ref
            }
            for idx, (item, summary, ref) in enumerate(
                zip(test_data, all_model_summaries[model_name], references)
            )
        ], f, indent=2, ensure_ascii=False)
    print(f"  ✅ Saved {output_path}")

# Save ensemble summaries
for strategy in ["voting", "weighted", "best_model", "ranking"]:
    output_path = os.path.join(OUTPUT_DIR, f"ensemble_{strategy}_summaries.json")
    with open(output_path, 'w', encoding='utf8') as f:
        json.dump([
            {
                "id": item.get("id", idx),
                "generated_summary": summary,
                "reference_summary": ref
            }
            for idx, (item, summary, ref) in enumerate(
                zip(test_data, ensemble_summaries[strategy], references)
            )
        ], f, indent=2, ensure_ascii=False)
    print(f"  ✅ Saved {output_path}")

# Save complete results
complete_results = {
    "experiment": "Ensemble Summarization - BART + LED + PEGASUS",
    "test_samples": len(test_data),
    "ensemble_weights": ENSEMBLE_WEIGHTS,
    "results": results,
    "best_approach": {
        "name": best_approach[0],
        "metrics": best_approach[1]
    }
}

results_path = os.path.join(OUTPUT_DIR, "ensemble_complete_results.json")
with open(results_path, 'w', encoding='utf8') as f:
    json.dump(complete_results, f, indent=2, ensure_ascii=False)

print(f"\n  ✅ Saved {results_path}")

# =========================================================
# GENERATE COMPARISON REPORT
# =========================================================

report_lines = []
report_lines.append("="*70)
report_lines.append("ENSEMBLE SUMMARIZATION EXPERIMENT")
report_lines.append("BART + LED + PEGASUS Fine-tuned Models")
report_lines.append("="*70)
report_lines.append("")
report_lines.append(f"Test Samples: {len(test_data)}")
report_lines.append(f"MMR Configuration: k={MMR_K} (BART/PEGASUS), k={MMR_K_LED} (LED)")
report_lines.append(f"Ensemble Weights: BART={ENSEMBLE_WEIGHTS['BART']}, LED={ENSEMBLE_WEIGHTS['LED']}, PEGASUS={ENSEMBLE_WEIGHTS['PEGASUS']}")
report_lines.append("")
report_lines.append("-"*70)
report_lines.append("INDIVIDUAL MODEL RESULTS")
report_lines.append("-"*70)

for model_name in ["BART", "LED", "PEGASUS"]:
    m = results[model_name]
    report_lines.append(f"\n{model_name}:")
    report_lines.append(f"  ROUGE-1:      {m['rouge1']:.4f}")
    report_lines.append(f"  ROUGE-2:      {m['rouge2']:.4f}")
    report_lines.append(f"  ROUGE-L:      {m['rougeL']:.4f}")
    report_lines.append(f"  BERTScore F1: {m['bertscore_f1']:.4f}")

report_lines.append("")
report_lines.append("-"*70)
report_lines.append("ENSEMBLE RESULTS")
report_lines.append("-"*70)

for strategy in ["voting", "weighted", "best_model", "ranking"]:
    m = results[f"ensemble_{strategy}"]
    report_lines.append(f"\nEnsemble-{strategy.upper()}:")
    report_lines.append(f"  ROUGE-1:      {m['rouge1']:.4f}")
    report_lines.append(f"  ROUGE-2:      {m['rouge2']:.4f}")
    report_lines.append(f"  ROUGE-L:      {m['rougeL']:.4f}")
    report_lines.append(f"  BERTScore F1: {m['bertscore_f1']:.4f}")

report_lines.append("")
report_lines.append("="*70)
report_lines.append(f"🏆 BEST APPROACH: {best_approach[0].upper()}")
report_lines.append("="*70)
report_lines.append(f"ROUGE-2:      {best_approach[1]['rouge2']:.4f}")
report_lines.append(f"BERTScore F1: {best_approach[1]['bertscore_f1']:.4f}")
report_lines.append("="*70)

report_text = "\n".join(report_lines)
print("\n" + report_text)

report_path = os.path.join(OUTPUT_DIR, "ensemble_report.txt")
with open(report_path, 'w', encoding='utf8') as f:
    f.write(report_text)

print(f"\n💾 Report saved to: {report_path}")

print("\n" + "="*70)
print("✅ ENSEMBLE EXPERIMENT COMPLETE")
print("="*70)
print(f"\nGenerated {len(results)} different approaches:")
print("  - 3 individual models (BART, LED, PEGASUS)")
print("  - 4 ensemble strategies (voting, weighted, best_model, ranking)")
print(f"\nBest performer: {best_approach[0]}")
print("="*70)


  [1/3] Loading BART...
  ✅ BART loaded - 406,290,432 parameters

  [2/3] Loading LED...
  ✅ LED loaded - 161,844,480 parameters

  [3/3] Loading PEGASUS...
  ✅ PEGASUS loaded - 570,797,056 parameters

✅ All models loaded successfully!

📂 Loading test data from test.json...
✅ Loaded 100 test samples

📊 Loading evaluation metrics...
✅ Metrics loaded: ROUGE, BERTScore

🚀 GENERATING SUMMARIES WITH ALL THREE MODELS


Processing samples: 100%|███████████████████████████████████████████████████████████| 100/100 [2:18:50<00:00, 83.30s/it]



✅ All summaries generated!

📊 EVALUATING ALL APPROACHES

  Evaluating BART...


/home/RSlab/.conda/envs/legal-nlp/lib/python3.10/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  ✅ BART - ROUGE-2: 0.1750, BERTScore F1: 0.8504

  Evaluating LED...
  ✅ LED - ROUGE-2: 0.2621, BERTScore F1: 0.8533

  Evaluating PEGASUS...
  ✅ PEGASUS - ROUGE-2: 0.1798, BERTScore F1: 0.8459

  Evaluating Ensemble-voting...
  ✅ Ensemble-voting - ROUGE-2: 0.1874, BERTScore F1: 0.8399

  Evaluating Ensemble-weighted...
  ✅ Ensemble-weighted - ROUGE-2: 0.1604, BERTScore F1: 0.8381

  Evaluating Ensemble-best_model...
  ✅ Ensemble-best_model - ROUGE-2: 0.2668, BERTScore F1: 0.8553

  Evaluating Ensemble-ranking...
  ✅ Ensemble-ranking - ROUGE-2: 0.2221, BERTScore F1: 0.8340

📊 COMPREHENSIVE RESULTS COMPARISON

Approach             ROUGE-1    ROUGE-2    ROUGE-L    BERTScore F1
----------------------------------------------------------------------
BART                 0.3523     0.1750     0.2047     0.8504      
LED                  0.4968     0.2621     0.2683     0.8533      
PEGASUS              0.3698     0.1798     0.2079     0.8459      
ensemble_voting      0.3864     0.1874     